<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 4: Chi Square

**VERİ BİLİMİ TEMELLERİ** · Modül 4 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta04/hafta04_chi_square.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta04/hafta04_chi_square.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 4 — Ki-Kare Bağımsızlık Testi

## Amaç

Ki-Kare (χ²) bağımsızlık testi, iki kategorik değişken arasında istatistiksel bir ilişki olup olmadığını test eder.

Bu defterde Titanic veri setini kullanarak:
1. **Bilet sınıfı** ile **hayatta kalma** arasındaki ilişkiyi
2. **Cinsiyet** ile **hayatta kalma** arasındaki ilişkiyi

inceleyeceğiz.

---

## 1. Kütüphanelerin Yüklenmesi

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `scipy` | Bilimsel hesaplama ve istatistik testleri |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
plt.rcParams['axes.unicode_minus'] = False

## 2. Titanic Veri Setinin Yüklenmesi

### Veri Setinin Yüklenmesi

Aşağıdaki kodda veri setini yüklüyoruz ve temel bilgilerine (boyut, sütunlar, ilk satırlar) bakıyoruz. Bu adım her veri bilimi projesinin başlangıcıdır.

In [ ]:
df = sns.load_dataset('titanic')
print(f"Veri seti boyutu: {df.shape}")
print(f"\nSütunlar: {list(df.columns)}")
df.head()

### Veri Gruplama ve Analiz

Verileri belirli kategorilere göre grupladıp özet istatistikler hesaplıyoruz. `groupby()` ile SQL'deki GROUP BY benzeri operasyonlar yapıyoruz.

In [ ]:
# Kullanacağımız sütunlar
print("Bilet Sınıfı Dağılımı:")
print(df['class'].value_counts())
print("\nHayatta Kalma Durumu:")
print(df['survived'].value_counts())
print("\nCinsiyet Dağılımı:")
print(df['sex'].value_counts())

## 3. Test 1: Bilet Sınıfı ile Hayatta Kalma İlişkisi

**Araştırma Sorusu:** Bilet sınıfı ile hayatta kalma arasında istatistiksel bir ilişki var mı?

**Hipotezler:**
- H₀: Bilet sınıfı ile hayatta kalma birbirinden bağımsızdır (ilişki yoktur).
- H₁: Bilet sınıfı ile hayatta kalma birbirinden bağımsız değildir (ilişki vardır).

### 3.1 Çapraz Tablo (Contingency Table)

In [ ]:
# Çapraz tablo oluşturma
capraz_tablo = pd.crosstab(df['class'], df['survived'], margins=True, margins_name='Toplam')
capraz_tablo.columns = ['Hayatını Kaybetti', 'Hayatta Kaldı', 'Toplam']
capraz_tablo.index = ['Birinci Sınıf', 'İkinci Sınıf', 'Üçüncü Sınıf', 'Toplam']
print("ÇAPRAZ TABLO: Bilet Sınıfı × Hayatta Kalma")
print("=" * 55)
capraz_tablo

### Yüzdelik çapraz tablo

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Yüzdelik çapraz tablo
capraz_yuzde = pd.crosstab(df['class'], df['survived'], normalize='index') * 100
capraz_yuzde.columns = ['Hayatını Kaybetti (%)', 'Hayatta Kaldı (%)']
capraz_yuzde.index = ['Birinci Sınıf', 'İkinci Sınıf', 'Üçüncü Sınıf']
print("YÜZDE TABLOSU (Satır bazlı)")
print("=" * 50)
capraz_yuzde.round(1)

### 3.2 Görselleştirme — Yığılmış Çubuk Grafiği

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Sayı bazlı ---
ct = pd.crosstab(df['class'], df['survived'])
ct.columns = ['Hayatını Kaybetti', 'Hayatta Kaldı']
ct.index = ['Birinci Sınıf', 'İkinci Sınıf', 'Üçüncü Sınıf']
ct.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], edgecolor='white', ax=axes[0])
axes[0].set_title('Bilet Sınıfına Göre Hayatta Kalma (Sayı)')
axes[0].set_xlabel('Bilet Sınıfı')
axes[0].set_ylabel('Yolcu Sayısı')
axes[0].legend(title='Durum')
axes[0].tick_params(axis='x', rotation=0)

# --- Yüzde bazlı ---
ct_pct = pd.crosstab(df['class'], df['survived'], normalize='index') * 100
ct_pct.columns = ['Hayatını Kaybetti', 'Hayatta Kaldı']
ct_pct.index = ['Birinci Sınıf', 'İkinci Sınıf', 'Üçüncü Sınıf']
ct_pct.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], edgecolor='white', ax=axes[1])
axes[1].set_title('Bilet Sınıfına Göre Hayatta Kalma (%)')
axes[1].set_xlabel('Bilet Sınıfı')
axes[1].set_ylabel('Yüzde (%)')
axes[1].legend(title='Durum')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 3.3 Ki-Kare Bağımsızlık Testi

### Çapraz tablo (margins olmadan)

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Çapraz tablo (margins olmadan)
ct_test = pd.crosstab(df['class'], df['survived'])

# Ki-Kare testi
chi2, p_value, dof, beklenen = stats.chi2_contingency(ct_test)

print("Kİ-KARE BAĞIMSIZLIK TESTİ")
print("Bilet Sınıfı × Hayatta Kalma")
print("=" * 50)
print(f"Ki-Kare İstatistiği (χ²): {chi2:.4f}")
print(f"p-değeri:                 {p_value:.6f}")
print(f"Serbestlik Derecesi (df): {dof}")
print()

if p_value < 0.05:
    print("KARAR: p < 0.05 → H₀ reddedilir.")
    print("Bilet sınıfı ile hayatta kalma arasında İSTATİSTİKSEL OLARAK")
    print("ANLAMLI bir ilişki vardır.")
else:
    print("KARAR: p >= 0.05 → H₀ reddedilemez.")
    print("Bilet sınıfı ile hayatta kalma arasında anlamlı bir ilişki yoktur.")

### 3.4 Beklenen Frekanslar Analizi

Ki-Kare testi, gözlenen frekansları beklenen frekanslarla karşılaştırır. Beklenen frekanslar, değişkenler bağımsız olsaydı gözlenmesi gereken değerlerdir.

In [ ]:
# Beklenen frekanslar tablosu
beklenen_df = pd.DataFrame(
    beklenen,
    index=['Birinci Sınıf', 'İkinci Sınıf', 'Üçüncü Sınıf'],
    columns=['Hayatını Kaybetti (Beklenen)', 'Hayatta Kaldı (Beklenen)']
)

print("BEKLENEN FREKANSLAR")
print("(Değişkenler bağımsız olsaydı gözlenmesi gereken değerler)")
print("=" * 60)
print(beklenen_df.round(2))
print()
print("GÖZLENEN FREKANSLAR")
print("=" * 60)
gozlenen_df = ct_test.copy()
gozlenen_df.index = ['Birinci Sınıf', 'İkinci Sınıf', 'Üçüncü Sınıf']
gozlenen_df.columns = ['Hayatını Kaybetti (Gözlenen)', 'Hayatta Kaldı (Gözlenen)']
print(gozlenen_df)

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Gözlenen vs Beklenen frekanslar görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

siniflar = ['Birinci\nSınıf', 'İkinci\nSınıf', 'Üçüncü\nSınıf']
x = np.arange(len(siniflar))
genislik = 0.35

# Hayatını kaybedenler
ax = axes[0]
ax.bar(x - genislik/2, ct_test.iloc[:, 0], genislik, label='Gözlenen', color='#e74c3c', alpha=0.8)
ax.bar(x + genislik/2, beklenen[:, 0], genislik, label='Beklenen', color='#e74c3c', alpha=0.4, edgecolor='#e74c3c', linewidth=2)
ax.set_title('Hayatını Kaybedenler: Gözlenen vs Beklenen')
ax.set_xticks(x)
ax.set_xticklabels(siniflar)
ax.set_ylabel('Frekans')
ax.legend()

# Hayatta kalanlar
ax = axes[1]
ax.bar(x - genislik/2, ct_test.iloc[:, 1], genislik, label='Gözlenen', color='#2ecc71', alpha=0.8)
ax.bar(x + genislik/2, beklenen[:, 1], genislik, label='Beklenen', color='#2ecc71', alpha=0.4, edgecolor='#2ecc71', linewidth=2)
ax.set_title('Hayatta Kalanlar: Gözlenen vs Beklenen')
ax.set_xticks(x)
ax.set_xticklabels(siniflar)
ax.set_ylabel('Frekans')
ax.legend()

plt.tight_layout()
plt.show()

## 4. Test 2: Cinsiyet ile Hayatta Kalma İlişkisi

**Araştırma Sorusu:** Cinsiyet ile hayatta kalma arasında istatistiksel bir ilişki var mı?

**Hipotezler:**
- H₀: Cinsiyet ile hayatta kalma birbirinden bağımsızdır.
- H₁: Cinsiyet ile hayatta kalma birbirinden bağımsız değildir.

### 4.1 Çapraz Tablo

In [ ]:
# Çapraz tablo
ct_cinsiyet = pd.crosstab(df['sex'], df['survived'])
ct_cinsiyet_gosterim = ct_cinsiyet.copy()
ct_cinsiyet_gosterim.columns = ['Hayatını Kaybetti', 'Hayatta Kaldı']
ct_cinsiyet_gosterim.index = ['Kadın', 'Erkek']

print("ÇAPRAZ TABLO: Cinsiyet × Hayatta Kalma")
print("=" * 45)
print(ct_cinsiyet_gosterim)
print()

# Yüzdelik
ct_pct2 = pd.crosstab(df['sex'], df['survived'], normalize='index') * 100
ct_pct2.columns = ['Hayatını Kaybetti (%)', 'Hayatta Kaldı (%)']
ct_pct2.index = ['Kadın', 'Erkek']
print("YÜZDE TABLOSU")
print("=" * 45)
print(ct_pct2.round(1))

### 4.2 Görselleştirme

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sayı bazlı
ct_viz = ct_cinsiyet.copy()
ct_viz.columns = ['Hayatını Kaybetti', 'Hayatta Kaldı']
ct_viz.index = ['Kadın', 'Erkek']
ct_viz.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], edgecolor='white', ax=axes[0])
axes[0].set_title('Cinsiyete Göre Hayatta Kalma (Sayı)')
axes[0].set_xlabel('Cinsiyet')
axes[0].set_ylabel('Yolcu Sayısı')
axes[0].legend(title='Durum')
axes[0].tick_params(axis='x', rotation=0)

# Yüzde bazlı
ct_pct_viz = pd.crosstab(df['sex'], df['survived'], normalize='index') * 100
ct_pct_viz.columns = ['Hayatını Kaybetti', 'Hayatta Kaldı']
ct_pct_viz.index = ['Kadın', 'Erkek']
ct_pct_viz.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], edgecolor='white', ax=axes[1])
axes[1].set_title('Cinsiyete Göre Hayatta Kalma (%)')
axes[1].set_xlabel('Cinsiyet')
axes[1].set_ylabel('Yüzde (%)')
axes[1].legend(title='Durum')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 4.3 Ki-Kare Testi

In [ ]:
chi2_2, p_value_2, dof_2, beklenen_2 = stats.chi2_contingency(ct_cinsiyet)

print("Kİ-KARE BAĞIMSIZLIK TESTİ")
print("Cinsiyet × Hayatta Kalma")
print("=" * 50)
print(f"Ki-Kare İstatistiği (χ²): {chi2_2:.4f}")
print(f"p-değeri:                 {p_value_2:.2e}")
print(f"Serbestlik Derecesi (df): {dof_2}")
print()

if p_value_2 < 0.05:
    print("KARAR: p < 0.05 → H₀ reddedilir.")
    print("Cinsiyet ile hayatta kalma arasında İSTATİSTİKSEL OLARAK")
    print("ANLAMLI bir ilişki vardır.")
else:
    print("KARAR: p >= 0.05 → H₀ reddedilemez.")
    print("Cinsiyet ile hayatta kalma arasında anlamlı bir ilişki yoktur.")

### Beklenen frekanslar

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Beklenen frekanslar
beklenen_df2 = pd.DataFrame(
    beklenen_2,
    index=['Kadın', 'Erkek'],
    columns=['Hayatını Kaybetti (Beklenen)', 'Hayatta Kaldı (Beklenen)']
)

print("BEKLENEN FREKANSLAR")
print("=" * 55)
print(beklenen_df2.round(2))
print()
print("GÖZLENEN FREKANSLAR")
print("=" * 55)
print(ct_cinsiyet_gosterim)

## 5. İki Testin Karşılaştırması

In [ ]:
karsilastirma = pd.DataFrame({
    'Test': ['Bilet Sınıfı × Hayatta Kalma', 'Cinsiyet × Hayatta Kalma'],
    'χ²': [chi2, chi2_2],
    'p-değeri': [p_value, p_value_2],
    'Serbestlik Derecesi': [dof, dof_2],
    'Sonuç': [
        'Anlamlı ilişki var' if p_value < 0.05 else 'İlişki yok',
        'Anlamlı ilişki var' if p_value_2 < 0.05 else 'İlişki yok'
    ]
}).set_index('Test')

print("İKİ TESTİN KARŞILAŞTIRMASI")
print("=" * 70)
karsilastirma

## 6. Sonuç ve Yorum

### Test 1: Bilet Sınıfı ve Hayatta Kalma

Ki-Kare testi sonucuna göre bilet sınıfı ile hayatta kalma arasında **istatistiksel olarak anlamlı** bir ilişki bulunmuştur. Birinci sınıf yolcuların hayatta kalma oranı, üçüncü sınıf yolculara göre çok daha yüksektir. Bu durum, cankurtaran botlarına erişim önceliğinin sınıfa göre belirlendiğini düşündürmektedir.

### Test 2: Cinsiyet ve Hayatta Kalma

Cinsiyet ile hayatta kalma arasında da **çok güçlü** bir istatistiksel ilişki bulunmuştur. Kadınların hayatta kalma oranı erkeklere göre çok daha yüksektir. Bu, "Kadınlar ve çocuklar önce" kuralının uygulandığını doğrulamaktadır.

### Ki-Kare Testinin Önemli Noktaları

1. Ki-Kare testi sadece **kategorik** değişkenler arasındaki ilişkiyi test eder.
2. Beklenen frekansların **5'ten büyük** olması gerekir (aksi halde Fisher'ın Kesin Testi kullanılmalıdır).
3. Ki-Kare istatistiği büyüdükçe, gözlenen ve beklenen frekanslar arasındaki fark da büyür.
4. p-değeri ne kadar küçükse, ilişkinin tesadüfi olma olasılığı o kadar düşüktür.

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://akademikyz.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>